# Fine-Tuning de SLMs para Detección de Botnets (LoRA)
Este notebook aplica la técnica de parametrización eficiente LoRA (Low-Rank Adaptation) a un conjunto de iteraciones de modelos SLM orientados a edge computing.

## 1. Instalación, Importación y Autenticación

In [ ]:
!pip install -q kagglehub transformers datasets peft accelerate scikit-learn psutil pandas numpy huggingface_hub

import kagglehub
import pandas as pd
import numpy as np
import os, time, psutil, gc, glob
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, matthews_corrcoef, precision_score, f1_score
from sklearn.preprocessing import RobustScaler
from transformers import AutoModel, AutoConfig
from peft import LoraConfig, get_peft_model, TaskType
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Autenticación para Modelos Gated (ej. Gemma)
from google.colab import userdata
from huggingface_hub import login
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Autenticación exitosa en HuggingFace.")
except Exception as e:
    print("⚠️ Advertencia: No se encontró HF_TOKEN. Modelos protegidos podrían fallar.")

## 2. Variables Globales e Hiperparámetros

In [ ]:
# ==========================================
# HIPERPARÁMETROS DEL BENCHMARK
# ==========================================
# Control del Dataset
MUESTRAS_BENIGN = 10000
MUESTRAS_ATAQUE = 3000
BATCH_SIZE_TRAIN = 64
BATCH_SIZE_TEST = 128
TEST_SIZE_SPLIT = 0.2

# Control de Entrenamiento
NUM_CLASSES = 3
EPOCHS = 3
LEARNING_RATE = 1e-4
NOISE_LEVEL = 0.01  # Data augmentation / Regularización

# Configuración Técnica de LoRA
LORA_R = 8
LORA_ALPHA = 32
LORA_TARGET_MODULES = "all-linear"

# Semilla de reproducibilidad
RANDOM_SEED = 42

# ==========================================
# VARIABLES GLOBALES
# ==========================================
# Características Universales Top 15 (Basado en Spearman + SHAP)
FEATURES_UNIVERSAL = [
    'HH_jit_L0.1_mean', 'MI_dir_L5_weight', 'H_L5_weight', 'HpHp_L0.01_radius',
    'HpHp_L0.01_std', 'HH_L0.1_radius', 'HH_L0.01_std', 'HH_L0.1_weight',
    'MI_dir_L0.1_mean', 'HH_jit_L5_mean', 'HpHp_L0.1_weight', 'HH_jit_L0.1_weight',
    'HH_L0.01_covariance', 'HpHp_L0.1_radius', 'HpHp_L5_magnitude'
]

# Selección de Modelos SLM
slm_benchmark_list = [
    # Familia BERT (Encoder-only)
    {"name": "DistilRoBERTa", "id": "distilroberta-base"},
    {"name": "DeBERTa-v3-Small", "id": "microsoft/deberta-v3-small"},
    {"name": "RoBERTa-Tiny", "id": "arampacha/roberta-tiny"},
    {"name": "BERT-Tiny", "id": "prajjwal1/bert-tiny"},
    {"name": "BERT-Mini", "id": "google/bert_uncased_L-4_H-256_A-4"},

    # Familia Generativa (Decoder-only)
    {"name": "Qwen2.5-0.5B", "id": "Qwen/Qwen2.5-0.5B-Instruct"},
    {"name": "TinyLlama-1.1B", "id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0"},
    {"name": "Phi-1.5", "id": "microsoft/phi-1_5"},
    {"name": "SmolLM-360M", "id": "HuggingFaceTB/SmolLM-360M-Instruct"},

    # Familia T5 (Encoder-Decoder)
    {"name": "Flan-T5-Small", "id": "google/flan-t5-small"},
    {"name": "TinyT5", "id": "google/t5-efficient-tiny"}
]

## 3. Arquitectura SLM Adaptada a Series Temporales (LoRA Pura)

In [ ]:
class SLMEdgeMulticlass(nn.Module):
    def __init__(self, checkpoint, num_features, num_classes=NUM_CLASSES):
        super().__init__()

        self.config = AutoConfig.from_pretrained(checkpoint, trust_remote_code=True)

        if getattr(self.config, "pad_token_id", None) is None:
            self.config.pad_token_id = getattr(self.config, "eos_token_id", 0)

        # Carga del modelo en precisión estándar
        self.transformer = AutoModel.from_pretrained(
            checkpoint,
            config=self.config,
            trust_remote_code=True
        )

        self.transformer = self.transformer.to(torch.float32)

        self.hidden_size = getattr(self.config, "hidden_size", getattr(self.config, "d_model", 768))
        self.embed_size = getattr(self.config, "embedding_size", self.hidden_size)

        # Capas de proyección y clasificación en float32
        self.feature_projector = nn.Linear(1, self.embed_size).to(torch.float32)
        self.feature_embeddings = nn.Parameter(torch.randn(1, num_features, self.embed_size, dtype=torch.float32))

        self.classifier = nn.Sequential(
            nn.LayerNorm(self.hidden_size),
            nn.Linear(self.hidden_size, num_classes)
        ).to(torch.float32)

    def forward(self, x, noise_level=0.0):
        target_dtype = torch.float32

        x = x.to(torch.float32)
        if self.training and noise_level > 0:
            x = x + torch.randn_like(x) * noise_level

        x = x.unsqueeze(-1)
        tokens = self.feature_projector(x) + self.feature_embeddings
        tokens = tokens.to(target_dtype)

        if getattr(self.config, "is_encoder_decoder", False):
            outputs = self.transformer.encoder(inputs_embeds=tokens)
        else:
            outputs = self.transformer(inputs_embeds=tokens)

        if hasattr(outputs, 'last_hidden_state'):
            hidden_states = outputs.last_hidden_state
        else:
            hidden_states = outputs[0]

        # Pooled a float32 para clasificación final
        pooled = hidden_states.mean(dim=1).to(torch.float32)
        return self.classifier(pooled)

## 4. Pipeline de Datos y Métricas

In [ ]:
def preparacion_de_datos(path):
    print(f"Extrayendo datos balanceados: {MUESTRAS_BENIGN} benignos / {MUESTRAS_ATAQUE} ataque...")

    def cargar_seguro(patron, label, limite):
        archivos = glob.glob(os.path.join(path, '**', patron), recursive=True)
        if not archivos: archivos = glob.glob(os.path.join(path, patron))
        if not archivos: return pd.DataFrame()

        muestras_por_archivo = limite // len(archivos)
        lista_dfs = []

        for f in archivos:
            try:
                total_rows = sum(1 for _ in open(f, 'r')) - 1
                if total_rows > muestras_por_archivo:
                    skip = sorted(np.random.choice(range(1, total_rows + 1), total_rows - muestras_por_archivo, replace=False))
                    df_temp = pd.read_csv(f, usecols=FEATURES_UNIVERSAL, skiprows=skip).dropna()
                else:
                    df_temp = pd.read_csv(f, usecols=FEATURES_UNIVERSAL).dropna()
                lista_dfs.append(df_temp)
            except: pass

        if lista_dfs:
            df = pd.concat(lista_dfs, ignore_index=True)
            df['label'] = label
            return df
        return pd.DataFrame()

    df_benign = cargar_seguro('*.benign.csv', 0, MUESTRAS_BENIGN)
    df_mirai = cargar_seguro('*.mirai.*.csv', 1, MUESTRAS_ATAQUE)
    df_gafgyt = cargar_seguro('*.gafgyt.*.csv', 2, MUESTRAS_ATAQUE)

    df_total = pd.concat([df_benign, df_mirai, df_gafgyt], ignore_index=True)
    labels = df_total.pop('label').values

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        df_total.values, labels, test_size=TEST_SIZE_SPLIT, random_state=RANDOM_SEED, stratify=labels)

    scaler = RobustScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    return (DataLoader(TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).long()), batch_size=BATCH_SIZE_TRAIN, shuffle=True),
            DataLoader(TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).long()), batch_size=BATCH_SIZE_TEST, shuffle=False))

def capturar_metricas(model, loader, device, nombre_modelo, fase, epoch):
    model.eval()
    y_true, y_pred = [], []
    mem_uso = psutil.Process().memory_info().rss / (1024 * 1024)
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters()) / (1024 * 1024)

    inicio = time.perf_counter()
    with torch.no_grad():
        for bx, by in loader:
            res = model(bx.to(device))
            y_pred.extend(torch.argmax(res, dim=1).cpu().numpy())
            y_true.extend(by.numpy())
    infer_ms = ((time.perf_counter() - inicio) / len(loader.dataset)) * 1000

    return {
        "Modelo": nombre_modelo, "Fase": fase, "Epoch": epoch,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average='macro', zero_division=0),
        "F1-Macro": f1_score(y_true, y_pred, average='macro', zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "RAM_MB": mem_uso, "Storage_MB": param_size, "Infer_ms": infer_ms
    }

## 5. Bucle de Entrenamiento Principal

In [ ]:
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
train_loader, test_loader = preparacion_de_datos(path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
historico_total = []

for slm in slm_benchmark_list:
    try:
        nombre = slm['name']
        estado_txt = "Full (LoRA Pura)"
        print(f"\n🚀 EVALUANDO: {nombre} | MODO: {estado_txt}")
        model = SLMEdgeMulticlass(slm['id'], len(FEATURES_UNIVERSAL), num_classes=NUM_CLASSES).to(device)

        # Inyectar LoRA parametrizado en la sección Global
        peft_config = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET_MODULES, task_type=TaskType.FEATURE_EXTRACTION
        )
        model.transformer = get_peft_model(model.transformer, peft_config)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(1, EPOCHS + 1):
            model.train()
            for bx, by in train_loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                outputs = model(bx, noise_level=NOISE_LEVEL)
                loss = criterion(outputs, by)
                loss.backward()
                optimizer.step()

            res = capturar_metricas(model, test_loader, device, nombre, estado_txt, epoch)
            historico_total.append(res)
            print(f"✅ LoRA - Epoch {epoch} | F1: {res['F1-Macro']:.4f} | RAM: {res['RAM_MB']:.2f} MB")

        # Guardar localmente el adaptador LoRA antes de limpiarlo de RAM
        import os
        os.makedirs(f"../modelos_entrenados/{nombre}_lora", exist_ok=True)
        model.save_pretrained(f"../modelos_entrenados/{nombre}_lora")
        print(f"\U0001f4be Guardado: ../modelos_entrenados/{nombre}_lora")
        
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ Error en {slm['name']}: {str(e)}")
        gc.collect()
        torch.cuda.empty_cache()

df_final = pd.DataFrame(historico_total)
df_final.to_csv("benchmark_iot_lora.txt", sep="\t", index=False)
print("\n✅ Benchmark consolidado completado.")

## 6. Visualización de Resultados

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Preparar los datos leyendo del archivo generado
df = pd.read_csv('benchmark_iot_lora.txt', sep='\t')

# Extraer solo los resultados del último Epoch
df = df[df['Epoch'] == df['Epoch'].max()].copy()

# Renombrar las columnas para que coincidan con la estructura gráfica deseada
df = df.rename(columns={
    'Fase': 'Tipo',
    'RAM_MB': 'RAM (MB)',
    'Storage_MB': 'Storage (MB)',
    'Infer_ms': 'Inferencia (ms)'
})

# Limpiar el tipo para abreviarlo temporalmente si es necesario, o dejarlo como está.
# Crear la columna identificadora combinada para los ejes
df['Modelo_Modo'] = df['Modelo'] + ' (' + df['Tipo'] + ')'

# Ordenar por el "más importante" (Accuracy) de mayor a menor para los gráficos
df = df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

# Configuración de estilo
sns.set_theme(style="whitegrid")
# Aumentar el número de filas de subplots a 4 y ajustar la altura de la figura
fig, axes = plt.subplots(4, 1, figsize=(14, 28))

# --- GRÁFICO 1: MÉTRICAS DE CALIDAD (Accuracy, F1, MCC) ---
df_melted = df.melt(id_vars='Modelo_Modo', value_vars=['Accuracy', 'F1-Macro', 'MCC'],
                    var_name='Métrica', value_name='Valor')

sns.barplot(data=df_melted, x='Modelo_Modo', y='Valor', hue='Métrica', ax=axes[0], palette='viridis')
axes[0].set_title('Comparativa de Calidad: Accuracy, F1-Macro y MCC (Ordenado por Accuracy)', fontsize=16, fontweight='bold')
axes[0].set_ylim(0.89, 1.0)  # Zoom para resaltar diferencias en modelos de alta precisión
axes[0].set_ylabel('Puntuación')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(loc='upper right')

# --- GRÁFICO 2: EFICIENCIA (VELOCIDAD VS ALMACENAMIENTO) (Scatter Plot original) ---
sns.scatterplot(data=df, x='Inferencia (ms)', y='Storage (MB)', hue='Modelo_Modo',
                size='Accuracy', sizes=(100, 500), ax=axes[1], palette='tab20')

for i in range(df.shape[0]):
    axes[1].text(df['Inferencia (ms)'][i] + 0.1, df['Storage (MB)'][i],
                 df['Modelo_Modo'][i], fontsize=8, verticalalignment='center')

axes[1].set_title('Eficiencia (Scatter Plot Original): Tiempo de Inferencia vs Peso en Disco', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Inferencia (ms)')
axes[1].set_ylabel('Almacenamiento (MB)')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='Modelos')

# --- GRÁFICO 3: CONSUMO DE RAM ---
df_ram_sorted = df.sort_values('RAM (MB)')
sns.barplot(data=df_ram_sorted, x='Modelo_Modo', y='RAM (MB)', ax=axes[2], palette='magma')
axes[2].set_title('Consumo de Memoria RAM durante el proceso', fontsize=16, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set_ylabel('RAM (MB)')

# --- GRÁFICO 4: NUEVA GRÁFICA DE EFICIENCIA LIMITADA (Inferencia vs Almacenamiento con ZOOM) ---
sns.scatterplot(data=df, x='Inferencia (ms)', y='Storage (MB)', hue='Modelo_Modo',
                size='Accuracy', sizes=(200, 800), ax=axes[3], palette='rocket_r', legend=False)

for i in range(df.shape[0]):
    # Límites para limpieza
    if (0 <= df['Inferencia (ms)'][i] <= 3) and (0 <= df['Storage (MB)'][i] <= 1000):
        modo_corto = f"({df['Tipo'][i][0]})"
        label_corta = f"{df['Modelo'][i]} {modo_corto}"
        axes[3].text(df['Inferencia (ms)'][i] + 0.05, df['Storage (MB)'][i] + 30,
                     label_corta, fontsize=9, verticalalignment='center')

axes[3].set_xlim(-0.5, 2.5)
axes[3].set_ylim(-100, 800)

axes[3].set_title('Eficiencia (Zoom): Inferencia (ms) vs Almacenamiento (MB) para Modelos Ligeros', fontsize=16, fontweight='bold')
axes[3].set_xlabel('Inferencia (ms)')
axes[3].set_ylabel('Almacenamiento (MB)')

plt.tight_layout()
plt.savefig('comparativa_completa_modelos_4_graficos.png')
plt.show()